In [1]:
import pandas as pd
from datasets import Dataset, DatasetDict, ClassLabel

# Train CSV: title, full_text, generated
df = pd.read_csv("../open/train.csv")

df['full_text'] = df['full_text'].str.split('\n')

df = (
    df[['title','full_text', 'generated']]
    .explode('full_text')
    .rename(columns={'full_text': 'full_text'})
    .reset_index(drop=True)
)

df = df.rename(columns={"generated":"label"})



df['char_len'] = df['full_text'].str.len()


top183_idx = df.nlargest(400, 'char_len').index


df = df.drop(index=top183_idx).reset_index(drop=True)
y = df['label']


ds = Dataset.from_pandas(df[["full_text", "label"]])


ds = ds.cast_column("label", ClassLabel(names=["human", "ai"]))  # 0=human, 1=ai


ds_split = ds.train_test_split(test_size=0.1,
                               stratify_by_column="label",
                               seed=42)

dataset = DatasetDict({"train": ds_split["train"],
                       "eval":  ds_split["test"]})


Casting the dataset:   0%|          | 0/1225964 [00:00<?, ? examples/s]

In [2]:
from transformers import ElectraTokenizer, ElectraForSequenceClassification

MODEL_NAME = "monologg/koelectra-base-v3-discriminator"  
tokenizer  = ElectraTokenizer.from_pretrained(MODEL_NAME)
model      = ElectraForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2         
)

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
def tokenize_batch(batch):
    return tokenizer(
        batch["full_text"],
        padding="max_length",
        truncation=True,
        max_length=512,
    )

dataset = dataset.map(tokenize_batch, batched=True, remove_columns=["full_text"], num_proc=20)
dataset.set_format(type="torch",
                   columns=["input_ids", "attention_mask", "label"])
dataset.save_to_disk("Ada")

Map (num_proc=20):   0%|          | 0/1103367 [00:00<?, ? examples/s]

Map (num_proc=20):   0%|          | 0/122597 [00:00<?, ? examples/s]

Saving the dataset (0/7 shards):   0%|          | 0/1103367 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/122597 [00:00<?, ? examples/s]

In [4]:
from transformers import Trainer
import torch
import torch.nn.functional as F
from superloss import HardFirstSuperLoss

class SuperLossTrainer(Trainer):
    hfs = HardFirstSuperLoss(
        2, 0.7, 0.1
    )
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """
        Custom loss function implementing SuperLoss weighting
        **kwargs added to handle additional arguments like num_items_in_batch
        """
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")  # (B, C)
        
        base_loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            labels.view(-1),
            reduction='none'
        )  # (B,)

        # 🔥 Weighted loss
        spl, sig = self.hfs(base_loss)
        super_loss = spl.mean()
        
        return (super_loss, outputs) if return_outputs else super_loss

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # << GPU 1번만 사용하도록 제한

from transformers import TrainingArguments
import torch
from sklearn.metrics import roc_auc_score, log_loss

import torch
device = torch.device("cuda:0")  # 내부적으로는 0번으로 보입니다
model.to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        "auc": roc_auc_score(labels, probs),
        "log_loss": log_loss(labels, probs),
    }

args = TrainingArguments(
    output_dir="koelectra-detector_Ada",
    per_device_train_batch_size=78,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=1000,
    logging_steps=200,
    learning_rate=2e-5,
    num_train_epochs=10,
    load_best_model_at_end=True,
    metric_for_best_model="auc",
    # fp16=True,
    bf16=True,

    # lr_scheduler_type="linear",
    # warmup_steps=500,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

/tmp/ipykernel_3576217/4067844536.py:37: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,Auc,Log Loss,Runtime,Samples Per Second,Steps Per Second
500,0.226000,0.216077,0.720643,0.216074,370.916500,330.525000,5.166000
1000,0.216700,0.210604,0.737820,0.210606,370.991100,330.458000,5.165000
1500,0.203000,0.223682,0.741986,0.223676,370.511100,330.886000,5.171000
2000,0.203200,0.212273,0.743158,0.212270,370.658000,330.755000,5.169000
2500,0.205600,0.231370,0.744156,0.231376,370.584200,330.821000,5.170000
3000,0.199100,0.216292,0.745988,0.216302,370.800300,330.628000,5.167000
3500,0.198400,0.201455,0.747733,0.201435,370.627900,330.782000,5.170000
4000,0.201100,0.206047,0.747221,0.206017,370.805900,330.623000,5.167000
4500,0.195800,0.196616,0.746675,0.196624,371.086900,330.373000,5.163000
5000,0.198200,0.209087,0.747658,0.209079,370.517700,330.880000,5.171000


In [ ]:
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES')}")
print(f"torch.cuda.device_count(): {torch.cuda.device_count()}")
print(f"Current device: {torch.cuda.current_device()}")

CUDA_VISIBLE_DEVICES: 1
torch.cuda.device_count(): 1
Current device: 0


In [ ]:
print(trainer.state.best_model_checkpoint)
# ▶ koelectra-detector/checkpoint-1000   같은 경로가 출력됨
raise

koelectra-detector_3/checkpoint-400


RuntimeError: No active exception to reraise

In [9]:
import pandas as pd
from datasets import Dataset
from torch.utils.data import DataLoader
import torch
from transformers import ElectraTokenizer, ElectraForSequenceClassification

# ---------------- 0. 환경 & 학습이 끝난 모델 ----------------
MODEL_NAME = "monologg/koelectra-base-v3-discriminator"
tokenizer  = ElectraTokenizer.from_pretrained(MODEL_NAME)
model      = ElectraForSequenceClassification.from_pretrained(
    "koelectra-detector_4/checkpoint-4000"    # <─ Trainer가 저장한 베스트 체크포인트 경로
).eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# ---------------- 1. Test CSV 로드 ----------------
test = pd.read_csv("open/test.csv")            # ID, title, paragraph_index, paragraph_text
test = test.rename(columns={"paragraph_text": "full_text"})   # ← 사용 중인 전처리와 맞춤
X_test = Dataset.from_pandas(test[["full_text"]].rename(columns={"full_text": "text"}))

# ---------------- 2. 동일 토크나이저 전처리 ----------------
def tok_fn(batch):
    return tokenizer(batch["text"], padding="max_length",
                     truncation=True, max_length=512)

X_test = X_test.map(tok_fn, batched=True, remove_columns=["text"])
X_test.set_format(type="torch", columns=["input_ids", "attention_mask"])

loader = DataLoader(X_test, batch_size=32)

# ---------------- 3. 배치 추론 ----------------
probs = []
with torch.no_grad():
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(**batch).logits
        probs.extend(torch.softmax(logits, dim=-1)[:, 1].tolist())   # 1 == AI 확률

# ---------------- 4. sample_submission 채우기 ----------------
sub = pd.read_csv("open/sample_submission.csv", encoding="utf-8-sig")  # ID, generated
sub["generated"] = probs                                          # 열 이름 그대로 사용
sub.to_csv("3rd.csv", index=False, encoding="utf-8-sig")

print("✅  3rd.csv 저장 완료!")

Map:   0%|          | 0/1962 [00:00<?, ? examples/s]

✅  3rd.csv 저장 완료!
